# 🧠 Training a CNN on CIFAR-10
This notebook demonstrates how to train a Convolutional Neural Network (CNN) on the CIFAR-10 dataset using TensorFlow and Keras.

We will break down the process into clear, easy-to-understand steps.

## 1. Import Libraries
First, we import the necessary libraries. We need `tensorflow` for the model, `numpy` for array manipulation, and `matplotlib` for plotting.

In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")

## 2. Load and Preprocess Data
We load the CIFAR-10 dataset directly from Keras. The dataset contains 60,000 32x32 color images in 10 classes.

After loading, we normalize the pixel values to be between 0 and 1 by dividing by 255.0. This helps the neural network converge faster.

In [ ]:
# Load dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

print(f"Train shape: {x_train.shape}")
print(f"Test shape: {x_test.shape}")

# Normalize images to [0, 1]
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

## 3. Build the CNN Model
We use a `Sequential` model with alternating `Conv2D` and `MaxPooling2D` layers to extract features, followed by `Dense` layers for classification. 

We also add `Dropout` to prevent overfitting.

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
])

model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(32, 32, 3)),
    data_augmentation,
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

## 4. Train the Model
Now we train the model on our training data. We use 8 epochs and a batch size of 64. You can adjust these values to see how they affect training.

In [ ]:
epochs = 30
batch_size = 64

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=4,
    restore_best_weights=True
)

print(f"Training model for up to {epochs} epochs with Early Stopping...")
history = model.fit(
    x_train, y_train,
    epochs=epochs,
    batch_size=batch_size,
    validation_data=(x_test, y_test),
    callbacks=[early_stopping]
)

## 5. Evaluate and Save Model
After training, we evaluate the model's accuracy on the test dataset. Then, we save the trained model to the `models` folder so our Streamlit app can use it later.

In [ ]:
test_loss, test_acc = model.evaluate(x_test,  y_test, verbose=2)
print(f"\nFinal test accuracy: {test_acc:.4f}")

# Create models directory if it doesn't exist
os.makedirs('models', exist_ok=True)

model_path = os.path.join('models', 'cnn_cifar10_model.keras')
model.save(model_path)
print(f"Model saved to {model_path}")

## 6. Plot Results and Sample Predictions
Finally, we visualize the training process by plotting the accuracy and loss over time. We also plot some sample predictions from the test set.

In [ ]:
os.makedirs('assets', exist_ok=True)

# Plot Training and Validation Accuracy
plt.figure(figsize=(8, 6))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label = 'Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0, 1])
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')
plt.savefig(os.path.join('assets', 'training_accuracy.png'))
plt.show()

# Plot Training and Validation Loss
plt.figure(figsize=(8, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label = 'Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.savefig(os.path.join('assets', 'training_loss.png'))
plt.show()

# Show Sample Predictions
class_names = ["airplane", "automobile", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]
plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(x_test[i])
    
    # Predict
    pred = model.predict(np.expand_dims(x_test[i], axis=0), verbose=0)
    pred_label = class_names[np.argmax(pred)]
    true_label = class_names[y_test[i][0]]
    
    # Color green if correct, red if incorrect
    color = 'green' if pred_label == true_label else 'red'
    plt.xlabel(f"{pred_label} ({true_label})", color=color)

plt.tight_layout()
plt.savefig(os.path.join('assets', 'sample_predictions.png'))
plt.show()